# LRA ListOps: Comparación de arquitecturas (DDP multi-GPU con TorchDistributor)

Notebook para el benchmark **LRA ListOps** (clasificación de secuencias).
Compara 8 arquitecturas: FFN, LSTM, Transformer (learned PE), Transformer+RoPE,
Transformer+ALiBi, Transformer+RoPE+MoE, Mamba y Jamba.

Adaptado del notebook de WikiText para tarea de clasificación con accuracy como métrica.

## 0) Setup

In [0]:
#Check entorno

import sys, torch
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CXX11 ABI:", getattr(torch._C, "_GLIBCXX_USE_CXX11_ABI", "n/a"))

import subprocess
print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,"]).decode())

Python: 3.10.12 (main, Mar  3 2026, 11:56:32) [GCC 11.4.0]
Torch: 2.0.1+cu118
Torch CUDA: 11.8
CXX11 ABI: False
name, driver_version
Tesla V100-SXM2-32GB, 580.126.09
Tesla V100-SXM2-32GB, 580.126.09
Tesla V100-SXM2-32GB, 580.126.09
Tesla V100-SXM2-32GB, 580.126.09
Tesla V100-SXM2-32GB, 580.126.09
Tesla V100-SXM2-32GB, 580.126.09
Tesla V100-SXM2-32GB, 580.126.09
Tesla V100-SXM2-32GB, 580.126.09



In [0]:
dbutils.library.restartPython()

In [0]:
%pip install --no-build-isolation "mamba-ssm[causal-conv1d]==2.2.2"
dbutils.library.restartPython()

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
  Using cached mamba_ssm-2.2.2-cp310-cp310-linux_x86_64.whl
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


In [0]:
import mamba_ssm
from mamba_ssm import Mamba
print("mamba_ssm OK:", getattr(mamba_ssm, "__version__", "unknown"))

2026-05-23 11:24:49.613438: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-23 11:24:49.613494: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-23 11:24:49.613526: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-23 11:24:49.622490: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


mamba_ssm OK: 2.2.2


## 1) Config

In [0]:
import os, math, time, random
from dataclasses import dataclass
from typing import Optional, Dict, List
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import AutoTokenizer

import mlflow

# --- Mamba LM ---
try:
    from mamba_ssm import Mamba
except Exception as e:
    Mamba = None
    _mamba_import_error = e

# Example values to try: 2, 4, maybe 8 (depends on CPU cores)
os.environ["OMP_NUM_THREADS"] = "1"
print("OMP_NUM_THREADS =", os.environ.get("OMP_NUM_THREADS"), flush=True)


try:
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    os.environ["DATABRICKS_HOST"] = ctx.apiUrl().get()
    os.environ["DATABRICKS_TOKEN"] = ctx.apiToken().get()
    os.environ["MLFLOW_ENABLE_DB_SDK"] = "true"
except Exception:
    # Si por algún motivo no hay contexto (raro en Jobs), se ignora
    pass

# Guardar modelos en subcarpeta local del working dir
SAVE_DIR = os.path.join(os.getcwd(), "lra_models")
os.makedirs(SAVE_DIR, exist_ok=True)

OMP_NUM_THREADS = 1


In [0]:
from dataclasses import dataclass, field
from typing import Optional, Literal

@dataclass
class CFG:
    # =========================
    # Dataset / tokenización
    # =========================
    DATASET_NAME: str = "lra_benchmark"
    DATASET_CONFIG: str = "listops"
    CACHE_DIR: str = "/dbfs/cache/hf_lra_listops"

    TEXT_FIELD: str = "text"
    LABEL_FIELD: str = "label"

    TOKENIZER_TYPE: Literal["hf", "custom"] = "hf"

    # =========================
    # Longitud de secuencia (LRA)
    # =========================
    MAX_SEQ_LEN: int = 2048                  # {512, 1024, 2048}
    PAD_TO_MAX_LENGTH: bool = True
    TRUNCATION: bool = True
    LENGTH_MODE: Literal["fixed", "cap"] = "fixed"

    # =========================
    # Submuestreo / límites
    # =========================
    TRAIN_FRACTION: Optional[float] = None
    MAX_TRAIN_SAMPLES: Optional[int] = None   # dataset completo
    MAX_EVAL_SAMPLES: Optional[int] = None    # dataset completo

    # =========================
    # Batch sizing
    # =========================
    TOKENS_PER_GPU_TARGET: int = 8192
    BATCH_SIZE: Optional[int] = None

    # =========================
    # Entrenamiento (clasificación)
    # =========================
    SEED: int = 42
    EPOCHS: int = 10                          # suficiente para convergencia con dataset completo
    LR: float = 3e-4
    WEIGHT_DECAY: float = 0.01
    GRAD_CLIP: float = 1.0
    USE_AMP: bool = False                     # False: Mamba kernels incompatibles con AMP
    USE_GRAD_CHECKPOINTING: bool = True

    # =========================
    # Tarea / métrica
    # =========================
    NUM_LABELS: int = 10                      # ListOps: 10 clases (0..9)
    METRIC: Literal["accuracy", "f1"] = "accuracy"

    # =========================
    # Tamaños de modelo (base)
    # =========================
    D_MODEL: int = 256
    DROPOUT: float = 0.1

    # LSTM
    LSTM_LAYERS: int = 2

    # Transformer / ALiBi
    N_HEADS: int = 8
    N_LAYERS: int = 4
    FFN_DIM: int = 1024

    # RoPE
    ROPE_BASE: int = 10000

    # =========================
    # Mamba
    # =========================
    MAMBA_LAYERS: int = 4
    MAMBA_D_STATE: int = 16
    MAMBA_D_CONV: int = 4
    MAMBA_EXPAND: int = 2

    # =========================
    # MoE
    # =========================
    MOE_NUM_EXPERTS: int = 4
    MOE_TOP_K: int = 2
    MOE_AUX_LOSS_COEFF: float = 0.01

    # =========================
    # Jamba
    # =========================
    JAMBA_LAYERS: int = 4
    JAMBA_ATTN_EVERY_N: int = 2               # M → A → M → A

    # =========================
    # Logging / guardado
    # =========================
    EXPERIMENT_NAME: str = "/Users/david.grana@boehringer-ingelheim.com/project_lra"
    RUN_NAME: str = ""    # se autoconstruye en recompute()
    SAVE_DIR: str = "/dbfs/tmp/lra_models"

    # =========================
    # Campos derivados
    # =========================
    WORLD_SIZE: int = field(init=False, default=1)
    MICRO_BATCH: int = field(init=False, default=0)
    GRAD_ACCUM_STEPS: int = field(init=False, default=1)
    GLOBAL_BATCH_SIZE: int = field(init=False, default=0)
    TOKENS_PER_STEP_PER_GPU: int = field(init=False, default=0)
    GLOBAL_TOKENS_PER_STEP: int = field(init=False, default=0)

    def recompute(self, world_size: int = 1) -> None:
        if self.BATCH_SIZE is None:
            self.BATCH_SIZE = max(1, self.TOKENS_PER_GPU_TARGET // self.MAX_SEQ_LEN)

        # Si BATCH_SIZE es demasiado grande para la GPU, se puede forzar
        # un micro_batch menor y acumular gradientes.
        self.MICRO_BATCH = self.BATCH_SIZE
        self.GRAD_ACCUM_STEPS = 1

        self.WORLD_SIZE = int(world_size)
        self.GLOBAL_BATCH_SIZE = int(self.MICRO_BATCH * self.GRAD_ACCUM_STEPS * self.WORLD_SIZE)
        self.TOKENS_PER_STEP_PER_GPU = int(self.MICRO_BATCH * self.GRAD_ACCUM_STEPS * self.MAX_SEQ_LEN)
        self.GLOBAL_TOKENS_PER_STEP = int(self.GLOBAL_BATCH_SIZE * self.MAX_SEQ_LEN)
        self.RUN_NAME = f"lra_listops_L{self.MAX_SEQ_LEN}_{self.TOKENS_PER_GPU_TARGET}"

    def summary(self) -> str:
        return (
            f"MAX_SEQ_LEN={self.MAX_SEQ_LEN} | TOKENS_PER_GPU_TARGET={self.TOKENS_PER_GPU_TARGET} | "
            f"BATCH_SIZE(per_gpu)={self.BATCH_SIZE} | WORLD_SIZE={self.WORLD_SIZE} | "
            f"GLOBAL_BATCH={self.GLOBAL_BATCH_SIZE} | TOKENS/step/gpu={self.TOKENS_PER_STEP_PER_GPU} | "
            f"GLOBAL_TOKENS/step={self.GLOBAL_TOKENS_PER_STEP} | "
            f"EPOCHS={self.EPOCHS} | USE_AMP={self.USE_AMP}"
        )

# ---- instancia ----
cfg = CFG()
cfg.recompute(world_size=1)
print(cfg.summary())


MAX_SEQ_LEN=2048 | TOKENS_PER_GPU_TARGET=8192 | BATCH_SIZE(per_gpu)=4 | WORLD_SIZE=1 | GLOBAL_BATCH=4 | TOKENS/step/gpu=8192 | GLOBAL_TOKENS/step=8192 | EPOCHS=10 | USE_AMP=False


In [0]:
# Sanity check: GPUs disponibles
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

CUDA available: True
GPU count: 8


## 2) Dataset + tokenización

In [0]:
import warnings
from datasets import load_dataset
from typing import Optional

# =========================================================
# Tokenizer a nivel de palabra para ListOps
# =========================================================

class ListOpsTokenizer:
    """
    Tokenizer ligero que hace split por espacios.
    Vocabulario construido a partir del train set (~17 tokens).
    Evita los 50K embeddings de GPT-2 que enmascaran las
    diferencias de backbone entre arquitecturas.
    """
    PAD = "<pad>"
    UNK = "<unk>"

    def __init__(self):
        self.token2id: dict[str, int] = {}
        self.id2token: dict[int, str] = {}
        self.pad_token_id: int = 0

    # --- construcción ---
    def build_vocab(self, texts):
        self.token2id = {self.PAD: 0, self.UNK: 1}
        for text in texts:
            if text is None:
                continue
            for tok in text.strip().split():
                if tok not in self.token2id:
                    self.token2id[tok] = len(self.token2id)
        self.id2token = {v: k for k, v in self.token2id.items()}
        return self

    def __len__(self):
        return len(self.token2id)

    # --- encode batch (compatible con datasets.map) ---
    def __call__(self, texts, max_length: int,
                 truncation: bool = True,
                 padding: str = "max_length",
                 return_attention_mask: bool = True):
        all_ids, all_masks = [], []
        unk_id = self.token2id[self.UNK]
        pad_id = self.pad_token_id
        for text in texts:
            tokens = (text or "").strip().split()
            ids = [self.token2id.get(t, unk_id) for t in tokens]
            if truncation and len(ids) > max_length:
                ids = ids[:max_length]
            mask = [1] * len(ids)
            if padding == "max_length":
                pad_n = max_length - len(ids)
                ids  += [pad_id] * pad_n
                mask += [0] * pad_n
            all_ids.append(ids)
            all_masks.append(mask)
        out = {"input_ids": all_ids}
        if return_attention_mask:
            out["attention_mask"] = all_masks
        return out


# =========================================================
# Carga del dataset
# =========================================================

warnings.filterwarnings(
    "ignore",
    message=r"During large dataset downloads*",
    category=UserWarning,
)

def load_listops_with_fallback(name: str, config_name: str, cache_dir: str):
    try:
        ds = load_dataset(name, config_name, cache_dir=cache_dir)
        print(f"Loaded: {name}/{config_name}")
        return ds, (name, config_name)
    except Exception as e:
        print(f"[WARN] No pude cargar {name}/{config_name} -> {e}")
        for fb_name in ["fengyang0317/listops-1000", "fengyang0317/listops-128"]:
            try:
                ds = load_dataset(fb_name, cache_dir=cache_dir)
                print(f"Loaded fallback: {fb_name}")
                return ds, (fb_name, None)
            except Exception as e2:
                print(f"[WARN] Fallback failed {fb_name} -> {e2}")
        raise RuntimeError("No pude cargar ningún dataset de ListOps.")

raw_ds, used_config = load_listops_with_fallback(
    cfg.DATASET_NAME, cfg.DATASET_CONFIG, cfg.CACHE_DIR
)
print("Using:", used_config)


def infer_fields(ds_split):
    cols = list(ds_split.column_names)
    text_candidates  = ["text", "Source", "sentence", "input", "sequence"]
    label_candidates = ["label", "Target", "targets", "y"]
    text_field  = next((c for c in text_candidates  if c in cols), None)
    label_field = next((c for c in label_candidates if c in cols), None)
    if text_field is None or label_field is None:
        raise ValueError(f"No pude inferir campos. Columnas={cols}")
    return text_field, label_field

TEXT_FIELD, LABEL_FIELD = infer_fields(raw_ds["train"])
print("TEXT_FIELD:", TEXT_FIELD, "| LABEL_FIELD:", LABEL_FIELD)


# =========================================================
# Construir vocabulario desde train
# =========================================================

tokenizer = ListOpsTokenizer().build_vocab(raw_ds["train"][TEXT_FIELD])
vocab_size = len(tokenizer)
print(f"Vocab ListOps: {vocab_size} tokens -> {list(tokenizer.token2id.keys())[:20]}...")


# =========================================================
# Tokenización
# =========================================================

def tokenize_fn(batch):
    texts = batch[TEXT_FIELD]
    cleaned = [(t.strip() if t else "") for t in texts]
    tok = tokenizer(
        cleaned,
        max_length=cfg.MAX_SEQ_LEN,
        truncation=True,
        padding=("max_length" if cfg.PAD_TO_MAX_LENGTH else False),
        return_attention_mask=True,
    )
    tok["labels"] = batch[LABEL_FIELD]
    return tok


def prepare_split(ds_split, split_name: str, max_samples: Optional[int]):
    if cfg.TRAIN_FRACTION is not None and split_name == "train":
        ds_split = ds_split.shuffle(seed=cfg.SEED)
        take_n = max(1, int(len(ds_split) * cfg.TRAIN_FRACTION))
        ds_split = ds_split.select(range(take_n))

    remove_cols = [c for c in ds_split.column_names if c not in [LABEL_FIELD]]
    tokenized = ds_split.map(
        tokenize_fn, batched=True,
        remove_columns=remove_cols, desc=f"tok {split_name}",
    )

    if max_samples is not None and len(tokenized) > max_samples:
        tokenized = tokenized.select(range(max_samples))

    tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return tokenized


train_ds = prepare_split(raw_ds["train"],      "train",      getattr(cfg, "MAX_TRAIN_SAMPLES", None))
valid_ds = prepare_split(raw_ds["validation"], "validation", getattr(cfg, "MAX_EVAL_SAMPLES",  None))
test_ds  = prepare_split(raw_ds["test"],       "test",       getattr(cfg, "MAX_EVAL_SAMPLES",  None))

print("Samples:", len(train_ds), len(valid_ds), len(test_ds))

[WARN] No pude cargar lra_benchmark/listops -> Couldn't find a dataset script at /Workspace/Users/david.grana@boehringer-ingelheim.com/side_project/vavava/lra_benchmark/lra_benchmark.py or any data file in the same directory. Couldn't find 'lra_benchmark' on the Hugging Face Hub either: FileNotFoundError: Dataset 'lra_benchmark' doesn't exist on the Hub. If the repo is private or gated, make sure to log in with `huggingface-cli login`.
Loaded fallback: fengyang0317/listops-1000
Using: ('fengyang0317/listops-1000', None)
TEXT_FIELD: Source | LABEL_FIELD: Target
Vocab ListOps: 19 tokens -> ['<pad>', '<unk>', '(', '[MIN', '1', ')', '[MED', '8', '0', '3', '6', ']', '7', '[SM', '4', '2', '9', '5', '[MAX']...


tok train:   0%|          | 0/96000 [00:00<?, ? examples/s]

tok validation:   0%|          | 0/2000 [00:00<?, ? examples/s]

tok test:   0%|          | 0/2000 [00:00<?, ? examples/s]

Samples: 96000 2000 2000


In [0]:
import numpy as np

lengths = [len(t.strip().split()) for t in raw_ds["train"][TEXT_FIELD][:5000]]
print(f"Longitud secuencias (n=5000 muestras de train):")
print(f"  min={min(lengths)}, max={max(lengths)}, mean={np.mean(lengths):.0f}, median={np.median(lengths):.0f}")
print(f"  p90={np.percentile(lengths, 90):.0f}, p95={np.percentile(lengths, 95):.0f}, p99={np.percentile(lengths, 99):.0f}")
print(f"  % > 512:  {100*np.mean(np.array(lengths) > 512):.1f}%")
print(f"  % > 1024: {100*np.mean(np.array(lengths) > 1024):.1f}%")
print(f"  % > 2048: {100*np.mean(np.array(lengths) > 2048):.1f}%")

Longitud secuencias (n=5000 muestras de train):
  min=1501, max=5995, mean=3074, median=2818
  p90=4855, p95=5338, p99=5863
  % > 512:  100.0%
  % > 1024: 100.0%
  % > 2048: 76.2%


## 3) Modelos

In [0]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# =========================================================
# Pooling helpers
# =========================================================

def last_token_pool(x: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    """x: [B,T,D], attention_mask: [B,T] (1=real, 0=pad) -> [B,D]"""
    last_idx = attention_mask.long().sum(dim=1) - 1
    last_idx = last_idx.clamp(min=0)
    return x[torch.arange(x.size(0), device=x.device), last_idx]

def mean_pool(x: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    """Mean pooling over valid tokens."""
    mask = attention_mask.unsqueeze(-1).to(x.dtype)
    return (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)


# =========================================================
# 1. FFN Classifier
# =========================================================

class FFNClassifier(nn.Module):
    def __init__(self, vocab_size: int, num_classes: int, d_model: int,
                 hidden_dim: int, dropout: float, pooling: str = "mean"):
        super().__init__()
        self.pooling = pooling
        self.emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, d_model),
            nn.Dropout(dropout),
        )
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        x = self.emb(input_ids)
        x = self.drop(x)
        if self.pooling == "last":
            pooled = last_token_pool(x, attention_mask)
        else:
            pooled = mean_pool(x, attention_mask)
        h = self.mlp(pooled)
        h = self.norm(h)
        return self.head(h)


# =========================================================
# 2. LSTM Classifier
# =========================================================

from torch.nn.utils.rnn import pack_padded_sequence

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size: int, num_classes: int, d_model: int,
                 hidden_size: int, n_layers: int, dropout: float,
                 bidirectional: bool = False):
        super().__init__()
        self.bidirectional = bidirectional
        self.emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        lstm_dropout = dropout if n_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=d_model, hidden_size=hidden_size,
            num_layers=n_layers, batch_first=True,
            dropout=lstm_dropout, bidirectional=bidirectional,
        )
        out_dim = hidden_size * (2 if bidirectional else 1)
        self.norm = nn.LayerNorm(out_dim)
        self.head = nn.Linear(out_dim, num_classes)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        x = self.emb(input_ids)
        x = self.drop(x)
        lengths = attention_mask.long().sum(dim=1).clamp(min=1)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        if self.bidirectional:
            h = torch.cat([h_n[-2], h_n[-1]], dim=-1)
        else:
            h = h_n[-1]
        h = self.norm(h)
        return self.head(h)


# =========================================================
# 3. Vanilla Transformer Classifier (bloques manuales, sin RoPE/ALiBi)
# =========================================================

class VanillaCausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.dropout = dropout
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor,
                causal: bool = True) -> torch.Tensor:
        B, T, D = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # Padding mask
        key_pad = (attention_mask == 0).unsqueeze(1).unsqueeze(2)  # [B,1,1,T]
        att = att.masked_fill(key_pad, float("-inf"))

        # Causal mask
        if causal:
            causal_mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
            att = att.masked_fill(causal_mask, float("-inf"))

        w = torch.softmax(att, dim=-1)
        w = F.dropout(w, p=self.dropout, training=self.training)
        y = w @ v
        y = y.transpose(1, 2).contiguous().view(B, T, D)
        return self.out(y)


class VanillaTransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, ffn_dim: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.att = VanillaCausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model), nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor,
                causal: bool = True) -> torch.Tensor:
        x = x + self.att(self.ln1(x), attention_mask, causal=causal)
        x = x + self.ff(self.ln2(x))
        return x


class TransformerDecoderOnlyClassifier(nn.Module):
    def __init__(self, vocab_size: int, num_classes: int, d_model: int, n_heads: int,
                 n_layers: int, ffn_dim: int, dropout: float, max_len: int,
                 causal: bool = True, pooling: str = "last"):
        super().__init__()
        self.causal = causal
        self.pooling = pooling
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            VanillaTransformerBlock(d_model, n_heads, ffn_dim, dropout)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        B, T = input_ids.shape
        pos_ids = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.tok_emb(input_ids) + self.pos_emb(pos_ids)
        x = self.drop(x)
        for blk in self.blocks:
            x = blk(x, attention_mask, causal=self.causal)
        x = self.ln_f(x)
        if self.pooling == "mean":
            pooled = mean_pool(x, attention_mask)
        else:
            pooled = last_token_pool(x, attention_mask)
        return self.head(pooled)


# =========================================================
# 4. RoPE helpers + Transformer RoPE Classifier
# =========================================================

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim: int, max_seq_len: int, base: int = 10000):
        super().__init__()
        assert head_dim % 2 == 0
        self.head_dim = head_dim
        self.max_seq_len = max_seq_len
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.register_buffer("_cos", None, persistent=False)
        self.register_buffer("_sin", None, persistent=False)

    def _build_cache(self, device, dtype):
        t = torch.arange(self.max_seq_len, device=device, dtype=torch.float32)
        freqs = torch.einsum("t,f->tf", t, self.inv_freq.to(device=device))
        self._cos = freqs.cos().to(dtype=dtype)[None, None, :, :]
        self._sin = freqs.sin().to(dtype=dtype)[None, None, :, :]

    def forward(self, seq_len: int, device, dtype):
        if (self._cos is None or self._cos.device != device
                or self._cos.dtype != dtype or self._cos.size(2) < seq_len):
            self._build_cache(device, dtype)
        return self._cos[:, :, :seq_len, :], self._sin[:, :, :seq_len, :]


def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack((x1 * cos - x2 * sin, x1 * sin + x2 * cos), dim=-1).flatten(-2)


class RoPESelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float,
                 max_seq_len: int, rope_base: int = 10000, causal: bool = True):
        super().__init__()
        assert d_model % n_heads == 0
        head_dim = d_model // n_heads
        assert head_dim % 2 == 0
        self.n_heads = n_heads
        self.head_dim = head_dim
        self.dropout = dropout
        self.causal = causal
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)
        self.rope = RotaryEmbedding(head_dim=head_dim, max_seq_len=max_seq_len, base=rope_base)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        B, T, D = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(seq_len=T, device=x.device, dtype=q.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        key_pad = (attention_mask == 0).unsqueeze(1).unsqueeze(2)
        att = att.masked_fill(key_pad, float("-inf"))
        if self.causal:
            causal_mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
            att = att.masked_fill(causal_mask, float("-inf"))

        w = torch.softmax(att, dim=-1)
        w = F.dropout(w, p=self.dropout, training=self.training)
        y = w @ v
        y = y.transpose(1, 2).contiguous().view(B, T, D)
        return self.out(y)


class TransformerRoPEBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, ffn_dim: int, dropout: float,
                 max_seq_len: int, rope_base: int, causal: bool = True):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.att = RoPESelfAttention(d_model, n_heads, dropout,
                                     max_seq_len=max_seq_len, rope_base=rope_base, causal=causal)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model), nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        x = x + self.att(self.ln1(x), attention_mask)
        x = x + self.ff(self.ln2(x))
        return x


class TransformerRoPEClassifier(nn.Module):
    def __init__(self, vocab_size: int, num_classes: int, d_model: int, n_heads: int,
                 n_layers: int, ffn_dim: int, dropout: float, max_seq_len: int,
                 rope_base: int = 10000, causal: bool = True, pooling: str = "last"):
        super().__init__()
        self.pooling = pooling
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerRoPEBlock(d_model, n_heads, ffn_dim, dropout,
                                 max_seq_len=max_seq_len, rope_base=rope_base, causal=causal)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        x = self.tok_emb(input_ids)
        x = self.drop(x)
        for blk in self.blocks:
            x = blk(x, attention_mask)
        x = self.ln_f(x)
        if self.pooling == "mean":
            pooled = mean_pool(x, attention_mask)
        else:
            pooled = last_token_pool(x, attention_mask)
        return self.head(pooled)


# =========================================================
# 5. ALiBi Classifier
# =========================================================

def _alibi_slopes(n_heads: int) -> torch.Tensor:
    def _get_slopes_power_of_2(n):
        start = 2.0 ** (-8.0 / n)
        return torch.pow(start, torch.arange(1, n + 1))
    if math.log2(n_heads).is_integer():
        return _get_slopes_power_of_2(n_heads)
    n = 2 ** math.floor(math.log2(n_heads))
    slopes = _get_slopes_power_of_2(n)
    extra = _get_slopes_power_of_2(2 * n)[0::2][: (n_heads - n)]
    return torch.cat([slopes, extra], dim=0)


class ALiBiSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float, causal: bool = True):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.dropout = dropout
        self.causal = causal
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)
        slopes = _alibi_slopes(n_heads)
        self.register_buffer("slopes", slopes, persistent=False)  # (H,)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        B, T, D = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # ALiBi bias
        positions = torch.arange(T, device=x.device, dtype=att.dtype)
        rel_pos = positions.unsqueeze(0) - positions.unsqueeze(1)  # (T,T)
        bias = self.slopes.view(1, -1, 1, 1) * rel_pos.unsqueeze(0).unsqueeze(0)  # (1,H,T,T)
        att = att + bias

        # Padding mask
        key_pad = (attention_mask == 0).unsqueeze(1).unsqueeze(2)
        att = att.masked_fill(key_pad, float("-inf"))

        # Causal mask
        if self.causal:
            causal_mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
            att = att.masked_fill(causal_mask, float("-inf"))

        w = torch.softmax(att, dim=-1)
        w = F.dropout(w, p=self.dropout, training=self.training)
        y = w @ v
        y = y.transpose(1, 2).contiguous().view(B, T, D)
        return self.out(y)


class TransformerALiBiBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, ffn_dim: int, dropout: float, causal: bool = True):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.att = ALiBiSelfAttention(d_model, n_heads, dropout, causal=causal)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model), nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        x = x + self.att(self.ln1(x), attention_mask)
        x = x + self.ff(self.ln2(x))
        return x


class TransformerALiBiClassifier(nn.Module):
    def __init__(self, vocab_size: int, num_classes: int, d_model: int, n_heads: int,
                 n_layers: int, ffn_dim: int, dropout: float,
                 causal: bool = True, pooling: str = "last"):
        super().__init__()
        self.pooling = pooling
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerALiBiBlock(d_model, n_heads, ffn_dim, dropout, causal=causal)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        x = self.tok_emb(input_ids)
        x = self.drop(x)
        for blk in self.blocks:
            x = blk(x, attention_mask)
        x = self.ln_f(x)
        if self.pooling == "mean":
            pooled = mean_pool(x, attention_mask)
        else:
            pooled = last_token_pool(x, attention_mask)
        return self.head(pooled)


# =========================================================
# 6. MoE (Router + FFN + Block + Classifier)
# =========================================================

class MoERouter(nn.Module):
    def __init__(self, d_model: int, num_experts: int, top_k: int):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.gate = nn.Linear(d_model, num_experts, bias=False)

    def forward(self, x: torch.Tensor):
        logits = self.gate(x)
        probs = torch.softmax(logits, dim=-1)
        top_k_w, top_k_idx = torch.topk(probs, self.top_k, dim=-1)
        top_k_w = top_k_w / (top_k_w.sum(dim=-1, keepdim=True) + 1e-9)

        with torch.no_grad():
            top1 = top_k_idx[..., 0]
            f = F.one_hot(top1, self.num_experts).float().mean(dim=(0, 1))
        p = probs.mean(dim=(0, 1))
        aux_loss = self.num_experts * (f * p).sum()
        return top_k_w, top_k_idx, aux_loss


class MoEFFN(nn.Module):
    def __init__(self, d_model: int, ffn_dim: int, num_experts: int, top_k: int, dropout: float):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.router = MoERouter(d_model, num_experts, top_k)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
                nn.Linear(ffn_dim, d_model), nn.Dropout(dropout),
            )
            for _ in range(num_experts)
        ])

    def forward(self, x: torch.Tensor):
        B, T, D = x.shape
        weights, indices, aux_loss = self.router(x)
        x_flat = x.view(B * T, D)
        w_flat = weights.view(B * T, self.top_k)
        i_flat = indices.view(B * T, self.top_k)
        out = torch.zeros_like(x_flat)

        for e in range(self.num_experts):
            mask = (i_flat == e)
            if not mask.any():
                continue
            token_mask = mask.any(dim=-1)
            tok_ids = token_mask.nonzero(as_tuple=True)[0]
            expert_out = self.experts[e](x_flat[tok_ids])
            w = (w_flat[tok_ids] * mask[tok_ids].float()).sum(dim=-1, keepdim=True)
            out[tok_ids] += w * expert_out

        return out.view(B, T, D), aux_loss


class TransformerRoPE_MoE_Block(nn.Module):
    def __init__(self, d_model: int, n_heads: int, ffn_dim: int, dropout: float,
                 max_seq_len: int, rope_base: int, num_experts: int, top_k: int, causal: bool = True):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.att = RoPESelfAttention(d_model, n_heads, dropout,
                                     max_seq_len=max_seq_len, rope_base=rope_base, causal=causal)
        self.ln2 = nn.LayerNorm(d_model)
        self.moe = MoEFFN(d_model, ffn_dim, num_experts, top_k, dropout)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor):
        x = x + self.att(self.ln1(x), attention_mask)
        moe_out, aux_loss = self.moe(self.ln2(x))
        x = x + moe_out
        return x, aux_loss


class TransformerRoPE_MoE_Classifier(nn.Module):
    def __init__(self, vocab_size: int, num_classes: int, d_model: int, n_heads: int,
                 n_layers: int, ffn_dim: int, dropout: float, max_seq_len: int,
                 rope_base: int = 10000, num_experts: int = 4, top_k: int = 2,
                 causal: bool = True, pooling: str = "last"):
        super().__init__()
        self.pooling = pooling
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerRoPE_MoE_Block(d_model, n_heads, ffn_dim, dropout,
                                      max_seq_len=max_seq_len, rope_base=rope_base,
                                      num_experts=num_experts, top_k=top_k, causal=causal)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, num_classes)
        self._aux_loss = 0.0

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        x = self.tok_emb(input_ids)
        x = self.drop(x)
        total_aux = 0.0
        for blk in self.blocks:
            x, aux = blk(x, attention_mask)
            total_aux = total_aux + aux
        self._aux_loss = total_aux / len(self.blocks)
        x = self.ln_f(x)
        if self.pooling == "mean":
            pooled = mean_pool(x, attention_mask)
        else:
            pooled = last_token_pool(x, attention_mask)
        return self.lm_head(pooled)


# =========================================================
# 7. Mamba Classifier (con residual + pre-norm)
# =========================================================

try:
    from mamba_ssm import Mamba
    _mamba_import_error = None
except ImportError as e:
    Mamba = None
    _mamba_import_error = e


class MambaClassifier(nn.Module):
    def __init__(self, vocab_size: int, num_classes: int, d_model: int, n_layers: int,
                 d_state: int, d_conv: int, expand: int, dropout: float, pooling: str = "last"):
        super().__init__()
        if Mamba is None:
            raise ImportError(f"mamba-ssm no disponible: {_mamba_import_error}")
        self.pooling = pooling
        self.emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
            for _ in range(n_layers)
        ])
        self.norms = nn.ModuleList([
            nn.LayerNorm(d_model) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        x = self.emb(input_ids)
        x = self.drop(x)
        for norm, blk in zip(self.norms, self.blocks):
            x = x + blk(norm(x))                  # pre-norm + residual
        x = self.norm(x)
        if self.pooling == "mean":
            pooled = mean_pool(x, attention_mask)
        else:
            pooled = last_token_pool(x, attention_mask)
        return self.head(pooled)


# =========================================================
# 8. Jamba Classifier (híbrido Mamba + Attention)
# =========================================================

class JambaBlock(nn.Module):
    def __init__(self, block_type: str, d_model: int, n_heads: int, ffn_dim: int,
                 dropout: float, max_seq_len: int, rope_base: int,
                 mamba_d_state: int, mamba_d_conv: int, mamba_expand: int):
        super().__init__()
        self.block_type = block_type
        if block_type == "attention":
            self.ln1 = nn.LayerNorm(d_model)
            self.att = RoPESelfAttention(d_model, n_heads, dropout,
                                         max_seq_len=max_seq_len, rope_base=rope_base, causal=True)
            self.ln2 = nn.LayerNorm(d_model)
            self.ff = nn.Sequential(
                nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
                nn.Linear(ffn_dim, d_model), nn.Dropout(dropout),
            )
        elif block_type == "mamba":
            self.ln = nn.LayerNorm(d_model)
            self.mamba = Mamba(d_model=d_model, d_state=mamba_d_state,
                              d_conv=mamba_d_conv, expand=mamba_expand)
        else:
            raise ValueError(f"block_type debe ser 'mamba' o 'attention', got '{block_type}'")

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        if self.block_type == "attention":
            x = x + self.att(self.ln1(x), attention_mask)
            x = x + self.ff(self.ln2(x))
        else:
            x = x + self.mamba(self.ln(x))
        return x


class JambaClassifier(nn.Module):
    """
    Jamba: híbrido Mamba + Attention.
    n_layers=4, attn_every_n=2 → M → A → M → A
    """
    def __init__(self, vocab_size: int, num_classes: int, d_model: int, n_heads: int,
                 n_layers: int, ffn_dim: int, dropout: float, max_seq_len: int,
                 rope_base: int = 10000, mamba_d_state: int = 16, mamba_d_conv: int = 4,
                 mamba_expand: int = 2, attn_every_n: int = 2, pooling: str = "last"):
        super().__init__()
        self.pooling = pooling
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList()
        for i in range(n_layers):
            btype = "attention" if ((i + 1) % attn_every_n == 0) else "mamba"
            self.blocks.append(JambaBlock(
                block_type=btype, d_model=d_model, n_heads=n_heads,
                ffn_dim=ffn_dim, dropout=dropout,
                max_seq_len=max_seq_len, rope_base=rope_base,
                mamba_d_state=mamba_d_state, mamba_d_conv=mamba_d_conv,
                mamba_expand=mamba_expand,
            ))
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, num_classes)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        x = self.tok_emb(input_ids)
        x = self.drop(x)
        for blk in self.blocks:
            x = blk(x, attention_mask)
        x = self.ln_f(x)
        if self.pooling == "mean":
            pooled = mean_pool(x, attention_mask)
        else:
            pooled = last_token_pool(x, attention_mask)
        return self.lm_head(pooled)

    def block_layout(self) -> str:
        return " -> ".join("A" if b.block_type == "attention" else "M" for b in self.blocks)


## 4) Loss

In [0]:
loss_fn = nn.CrossEntropyLoss()

def compute_loss(logits, labels):

    return loss_fn(logits, labels)

def collate(batch):
    input_ids = torch.stack([x["input_ids"] for x in batch])            # [B, L]
    attention_mask = torch.stack([x["attention_mask"] for x in batch])  # [B, L]
    labels = torch.stack([x["labels"] for x in batch]).long()           # [B]
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler

def make_dataloaders_ddp(rank: int, world_size: int):
    train_sampler = DistributedSampler(train_ds, num_replicas=world_size, rank=rank, shuffle=True, drop_last=True)
    val_sampler   = DistributedSampler(valid_ds, num_replicas=world_size, rank=rank, shuffle=False, drop_last=False)
    test_sampler  = DistributedSampler(test_ds,  num_replicas=world_size, rank=rank, shuffle=False, drop_last=False)

    train_loader = DataLoader(
        train_ds, batch_size=cfg.BATCH_SIZE, sampler=train_sampler,
        collate_fn=collate, drop_last=True
    )
    valid_loader = DataLoader(
        valid_ds, batch_size=cfg.BATCH_SIZE, sampler=val_sampler,
        collate_fn=collate, drop_last=False
    )
    test_loader  = DataLoader(
        test_ds, batch_size=cfg.BATCH_SIZE, sampler=test_sampler,
        collate_fn=collate, drop_last=False
    )
    return train_loader, valid_loader, test_loader, train_sampler

## 5) DDP + TorchDistributor

In [0]:
from pyspark.ml.torch.distributor import TorchDistributor

import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler
from datetime import timedelta

def ddp_setup():
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    dist.init_process_group("nccl", timeout=timedelta(minutes=3))
    dist.barrier(device_ids=[local_rank])
    return local_rank

def ddp_barrier(local_rank: int):
    dist.barrier(device_ids=[local_rank])

def ddp_cleanup():
    dist.destroy_process_group()

def is_main_process():
    return (not dist.is_available()) or (not dist.is_initialized()) or dist.get_rank() == 0


@torch.no_grad()
def evaluate_ddp(model, loader):
    model.eval()
    device = torch.cuda.current_device()

    total_loss_sum = 0.0
    total_correct = 0
    total_samples = 0
    total_tokens = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        logits = model(input_ids, attention_mask)
        loss = compute_loss(logits, labels)

        B = labels.size(0)
        preds = logits.argmax(dim=-1)
        correct = (preds == labels).sum().item()

        total_loss_sum += loss.item() * B
        total_correct += correct
        total_samples += B
        total_tokens += int(attention_mask.sum().item())

    tens = torch.tensor(
        [total_loss_sum, total_correct, total_samples, total_tokens],
        device=device, dtype=torch.float64
    )
    dist.all_reduce(tens, op=dist.ReduceOp.SUM)

    loss_avg = (tens[0] / tens[2]).item()
    acc = (tens[1] / tens[2]).item()
    return {"loss": float(loss_avg), "acc": float(acc),
            "tokens": int(tens[3].item()), "samples": int(tens[2].item())}


from tqdm.auto import tqdm

def train_one_model_ddp(
    model_name: str,
    model: nn.Module,
    train_loader,
    valid_loader,
    test_loader,
    train_sampler,
    local_rank: int,
):
    device = torch.device(f"cuda:{local_rank}")
    model = model.to(device)

    # Detección automática de MoE: solo MoEFFN / MoERouter tienen "MoE" en el nombre de clase.
    # find_unused_parameters=True es necesario porque con top_k < num_experts
    # algunos expertos no reciben tokens y sus gradientes quedan sin reducir.
    is_moe = any("MoE" in type(m).__name__ for m in model.modules())
    ddp_model = DDP(model, device_ids=[local_rank],
                    find_unused_parameters=is_moe)

    world_size = dist.get_world_size()
    tokens_seen = 0

    opt = torch.optim.AdamW(ddp_model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    opt.zero_grad(set_to_none=True)

    log_every = 50

    # Reset peak memory
    torch.cuda.reset_peak_memory_stats(device)

    for epoch in range(1, cfg.EPOCHS + 1):
        ddp_model.train()
        train_sampler.set_epoch(epoch)

        iterator = train_loader
        if is_main_process():
            iterator = tqdm(train_loader, desc=f"[{model_name}] epoch {epoch}/{cfg.EPOCHS}", leave=False)

        running_loss = torch.zeros((), device=device)
        running_acc = torch.zeros((), device=device)
        running_steps = 0
        t_last = time.time()
        tokens_since_last_log = 0

        for step, batch in enumerate(iterator, start=1):
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            step_tokens = int(attention_mask.sum().item()) * world_size
            tokens_seen += step_tokens
            tokens_since_last_log += step_tokens
            tokens_seen_m = int(tokens_seen // 1_000_000)

            logits = ddp_model(input_ids, attention_mask)
            loss = compute_loss(logits, labels)

            # MoE auxiliary loss
            raw_model = ddp_model.module if hasattr(ddp_model, "module") else ddp_model
            if hasattr(raw_model, "_aux_loss") and isinstance(raw_model._aux_loss, torch.Tensor):
                loss = loss + cfg.MOE_AUX_LOSS_COEFF * raw_model._aux_loss

            # Capturar loss sin escalar para logging
            loss_for_log = loss.detach()

            # Escalar loss para acumulación de gradientes
            loss = loss / cfg.GRAD_ACCUM_STEPS
            loss.backward()

            if step % cfg.GRAD_ACCUM_STEPS == 0:
                if cfg.GRAD_CLIP is not None:
                    nn.utils.clip_grad_norm_(ddp_model.parameters(), cfg.GRAD_CLIP)
                opt.step()
                opt.zero_grad(set_to_none=True)

            preds = logits.detach().argmax(dim=-1)
            acc = (preds == labels).float().mean()

            running_loss += loss_for_log
            running_acc += acc.detach()
            running_steps += 1

            if is_main_process() and (step % log_every == 0):
                t_now = time.time()
                elapsed = t_now - t_last

                step_ms = elapsed * 1000.0 / log_every
                mlflow.log_metric("ms_per_step", float(step_ms), step=tokens_seen_m)

                if elapsed > 0:
                    mlflow.log_metric("tokens_per_sec", tokens_since_last_log / elapsed, step=tokens_seen_m)

                peak_mb = torch.cuda.max_memory_allocated(device) / (1024 * 1024)
                mlflow.log_metric("peak_gpu_mb", peak_mb, step=tokens_seen_m)

                avg_loss = (running_loss / max(1, running_steps)).item()
                avg_acc  = (running_acc  / max(1, running_steps)).item()
                running_loss.zero_()
                running_acc.zero_()
                running_steps = 0

                iterator.set_postfix({"loss": f"{avg_loss:.4f}", "acc": f"{avg_acc:.3f}", "ms": f"{step_ms:.1f}"})
                mlflow.log_metric("train_loss", float(avg_loss), step=tokens_seen_m)
                mlflow.log_metric("train_acc",  float(avg_acc),  step=tokens_seen_m)

                t_last = t_now
                tokens_since_last_log = 0

        # Si quedan gradientes acumulados sin aplicar al final del epoch
        if step % cfg.GRAD_ACCUM_STEPS != 0:
            if cfg.GRAD_CLIP is not None:
                nn.utils.clip_grad_norm_(ddp_model.parameters(), cfg.GRAD_CLIP)
            opt.step()
            opt.zero_grad(set_to_none=True)

        # Validación global
        val = evaluate_ddp(ddp_model, valid_loader)

        if is_main_process():
            print(f"[{model_name}] epoch={epoch} val_loss={val['loss']:.4f} val_acc={val['acc']:.4f}")
            mlflow.log_metric("val_loss", float(val["loss"]), step=epoch)
            mlflow.log_metric("val_acc",  float(val["acc"]),  step=epoch)

    # Test global
    test_m = evaluate_ddp(ddp_model, test_loader)

    if is_main_process():
        final_peak_mb = torch.cuda.max_memory_allocated(device) / (1024 * 1024)
        mlflow.log_metric("final_peak_gpu_mb", final_peak_mb)

    return test_m, ddp_model


import traceback

def ddp_worker():
    import time, traceback, torch, warnings
    import torch.distributed as dist
    import mlflow
    from contextlib import nullcontext

    local_rank = ddp_setup()
    rank = dist.get_rank()
    world_size = dist.get_world_size()

    warnings.filterwarnings(
        "ignore",
        message=r'Field "model_name" has conflict with protected namespace "model_"*',
        category=UserWarning,
    )

    try:
        cfg.recompute(world_size=world_size)
    except TypeError:
        pass

    # Resumen de la configuración de batching efectiva
    if is_main_process():
        print(f"[cfg] MAX_SEQ_LEN={cfg.MAX_SEQ_LEN} | micro_batch={cfg.MICRO_BATCH} | "
              f"grad_accum={cfg.GRAD_ACCUM_STEPS} | "
              f"tokens/step/gpu={cfg.TOKENS_PER_STEP_PER_GPU} | "
              f"global_tokens/step={cfg.GLOBAL_TOKENS_PER_STEP}")

    # Dataloaders
    train_loader, valid_loader, test_loader, train_sampler = make_dataloaders_ddp(rank, world_size)

    # MLflow solo rank0
    if is_main_process():
        mlflow.set_experiment(cfg.EXPERIMENT_NAME)
        if mlflow.active_run() is not None:
            mlflow.end_run()

    # Crear directorio de guardado
    os.makedirs(cfg.SAVE_DIR, exist_ok=True)

    # Mamba disponibilidad
    mamba_ok = True
    try:
        from mamba_ssm import Mamba
    except Exception as e:
        mamba_ok = False
        if is_main_process():
            print(f"[WARN] Mamba no disponible, se omite. Error: {e}")

    num_classes = cfg.NUM_LABELS

    # === Todos los model builders ===
    model_builders = {
        "FFN": lambda: FFNClassifier(
            vocab_size=vocab_size, num_classes=num_classes,
            d_model=cfg.D_MODEL, hidden_dim=cfg.D_MODEL,
            dropout=cfg.DROPOUT, pooling="mean",
        ),

        "LSTM": lambda: LSTMClassifier(
            vocab_size=vocab_size, num_classes=num_classes,
            d_model=cfg.D_MODEL, hidden_size=cfg.D_MODEL,
            n_layers=cfg.LSTM_LAYERS, dropout=cfg.DROPOUT,
            bidirectional=False,
        ),

        "Transformer": lambda: TransformerDecoderOnlyClassifier(
            vocab_size=vocab_size, num_classes=num_classes,
            d_model=cfg.D_MODEL, n_heads=cfg.N_HEADS,
            n_layers=cfg.N_LAYERS, ffn_dim=cfg.FFN_DIM,
            dropout=cfg.DROPOUT, max_len=cfg.MAX_SEQ_LEN,
            causal=True, pooling="last",
        ),

        "Transformer_RoPE": lambda: TransformerRoPEClassifier(
            vocab_size=vocab_size, num_classes=num_classes,
            d_model=cfg.D_MODEL, n_heads=cfg.N_HEADS,
            n_layers=cfg.N_LAYERS, ffn_dim=cfg.FFN_DIM,
            dropout=cfg.DROPOUT, max_seq_len=cfg.MAX_SEQ_LEN,
            rope_base=cfg.ROPE_BASE, causal=True, pooling="last",
        ),

        "Transformer_ALiBi": lambda: TransformerALiBiClassifier(
            vocab_size=vocab_size, num_classes=num_classes,
            d_model=cfg.D_MODEL, n_heads=cfg.N_HEADS,
            n_layers=cfg.N_LAYERS, ffn_dim=cfg.FFN_DIM,
            dropout=cfg.DROPOUT, causal=True, pooling="last",
        ),

        "Transformer_RoPE_MoE": lambda: TransformerRoPE_MoE_Classifier(
            vocab_size=vocab_size, num_classes=num_classes,
            d_model=cfg.D_MODEL, n_heads=cfg.N_HEADS,
            n_layers=cfg.N_LAYERS, ffn_dim=cfg.FFN_DIM,
            dropout=cfg.DROPOUT, max_seq_len=cfg.MAX_SEQ_LEN,
            rope_base=cfg.ROPE_BASE,
            num_experts=cfg.MOE_NUM_EXPERTS, top_k=cfg.MOE_TOP_K,
            causal=True, pooling="last",
        ),
    }

    if mamba_ok:
        model_builders["Mamba"] = lambda: MambaClassifier(
            vocab_size=vocab_size, num_classes=num_classes,
            d_model=cfg.D_MODEL, n_layers=cfg.MAMBA_LAYERS,
            d_state=cfg.MAMBA_D_STATE, d_conv=cfg.MAMBA_D_CONV,
            expand=cfg.MAMBA_EXPAND, dropout=cfg.DROPOUT, pooling="last",
        )
        model_builders["Jamba"] = lambda: JambaClassifier(
            vocab_size=vocab_size, num_classes=num_classes,
            d_model=cfg.D_MODEL, n_heads=cfg.N_HEADS,
            n_layers=cfg.JAMBA_LAYERS, ffn_dim=cfg.FFN_DIM,
            dropout=cfg.DROPOUT, max_seq_len=cfg.MAX_SEQ_LEN,
            rope_base=cfg.ROPE_BASE,
            mamba_d_state=cfg.MAMBA_D_STATE, mamba_d_conv=cfg.MAMBA_D_CONV,
            mamba_expand=cfg.MAMBA_EXPAND, attn_every_n=cfg.JAMBA_ATTN_EVERY_N,
            pooling="last",
        )

    # === MLflow context ===
    parent_ctx = (mlflow.start_run(run_name=cfg.RUN_NAME) if is_main_process() else nullcontext())

    try:
        with parent_ctx:
            if is_main_process():
                mlflow.log_params({
                    "used_config": str(used_config),
                    "world_size": world_size,
                    "max_seq_len": cfg.MAX_SEQ_LEN,
                    "batch_size_per_gpu": cfg.BATCH_SIZE,
                    "micro_batch": cfg.MICRO_BATCH,
                    "grad_accum_steps": cfg.GRAD_ACCUM_STEPS,
                    "d_model": cfg.D_MODEL,
                    "n_layers": cfg.N_LAYERS,
                    "n_heads": cfg.N_HEADS,
                    "ffn_dim": cfg.FFN_DIM,
                    "lr": cfg.LR,
                    "weight_decay": cfg.WEIGHT_DECAY,
                    "epochs": cfg.EPOCHS,
                    "use_amp": cfg.USE_AMP,
                    "moe_num_experts": cfg.MOE_NUM_EXPERTS,
                    "moe_top_k": cfg.MOE_TOP_K,
                    "moe_aux_coeff": cfg.MOE_AUX_LOSS_COEFF,
                    "jamba_attn_every_n": cfg.JAMBA_ATTN_EVERY_N,
                })
                print(f"[rank0] Starting models: {list(model_builders.keys())}")

            ddp_barrier(local_rank)

            for name, builder in model_builders.items():
                ddp_barrier(local_rank)

                if is_main_process():
                    print(f"\n=== Training model: {name} ===")

                nested_ctx = (mlflow.start_run(run_name=name, nested=True) if is_main_process() else nullcontext())

                try:
                    with nested_ctx:
                        model = builder()

                        if is_main_process():
                            n_params = sum(p.numel() for p in model.parameters())
                            n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
                            mlflow.log_param("model_type", name)
                            mlflow.log_param("max_seq_len", cfg.MAX_SEQ_LEN)
                            mlflow.log_param("n_params_total", int(n_params))
                            mlflow.log_param("n_params_trainable", int(n_train))

                            # Parámetros activos por token (relevante solo para MoE)
                            if any("MoE" in type(m).__name__ for m in model.modules()):
                                moe_ffn_modules = [m for m in model.modules() if type(m).__name__ == "MoEFFN"]
                                # params por experto (todos iguales): tomamos el primero
                                params_per_expert = sum(
                                    p.numel() for p in moe_ffn_modules[0].experts[0].parameters()
                                )
                                # expertos inactivos por capa: (E - k)
                                inactive_per_layer = params_per_expert * (cfg.MOE_NUM_EXPERTS - cfg.MOE_TOP_K)
                                n_active = n_params - inactive_per_layer * len(moe_ffn_modules)
                                mlflow.log_param("n_params_active", int(n_active))
                            else:
                                mlflow.log_param("n_params_active", int(n_params))

                            if hasattr(model, "block_layout"):
                                mlflow.log_param("jamba_layout", model.block_layout())

                            if hasattr(model, "_aux_loss"):
                                mlflow.log_param("moe_num_experts", cfg.MOE_NUM_EXPERTS)
                                mlflow.log_param("moe_top_k", cfg.MOE_TOP_K)
                                mlflow.log_param("moe_aux_coeff", cfg.MOE_AUX_LOSS_COEFF)

                            print(f"[{name}] params total={n_params:,} trainable={n_train:,}")

                        start = time.time()
                        test_m, ddp_model = train_one_model_ddp(
                            name, model,
                            train_loader, valid_loader, test_loader, train_sampler,
                            local_rank=local_rank
                        )
                        elapsed = time.time() - start

                        if is_main_process():
                            mlflow.log_metric("test_loss", float(test_m["loss"]))
                            mlflow.log_metric("test_acc",  float(test_m["acc"]))
                            mlflow.log_metric("train_seconds", float(elapsed))

                            model_filename = f"model_{name}_L{cfg.MAX_SEQ_LEN}.pt"
                            model_path = os.path.join(cfg.SAVE_DIR, model_filename)
                            torch.save(ddp_model.module.state_dict(), model_path)
                            mlflow.log_artifact(model_path)

                            print(f"=== Done: {name} | test_loss={test_m['loss']:.4f} test_acc={test_m['acc']:.4f} ===")

                except Exception as e:
                    print(f"[rank{rank}] ERROR in model {name}: {e}")
                    print(traceback.format_exc())
                    raise
                finally:
                    try:
                        del ddp_model
                    except Exception:
                        pass
                    try:
                        del model
                    except Exception:
                        pass
                    torch.cuda.empty_cache()

                ddp_barrier(local_rank)

    finally:
        try:
            ddp_cleanup()
        except Exception:
            pass

    return None


## 6) Lanzar en 8 GPUs y mostrar resultados

In [0]:
_ = TorchDistributor(num_processes=8, local_mode=True, use_gpu=True).run(ddp_worker)
print("Entrenamiento distribuido finalizado. Revisa métricas/artefactos en MLflow.")


INFO:TorchDistributor:Started local training with 8 processes


master_addr is only used for static rdzv_backend and when rdzv_endpoint is not specified.
[cfg] MAX_SEQ_LEN=2048 | micro_batch=4 | grad_accum=1 | tokens/step/gpu=8192 | global_tokens/step=65536
Sat May 23 16:48:51 2026 Connection to spark from PID  73369
Sat May 23 16:48:51 2026 Initialized gateway on port 33877
Sat May 23 16:48:51 2026 Connected to spark.
[rank0] Starting models: ['FFN', 'LSTM', 'Transformer', 'Transformer_RoPE', 'Transformer_ALiBi', 'Transformer_RoPE_MoE', 'Mamba', 'Jamba']

=== Training model: FFN ===
[FFN] params total=139,530 trainable=139,530
[FFN] epoch 1/10:  78%|███████▊  | 2350/3000 [01:10<00:15, 40.66it/s, loss=2.2661, acc=

*** WARNING: max output size exceeded, skipping output. ***

[Transformer_ALiBi] epoch=8 val_loss=2.2585 val_acc=0.1600
